# Atualização da base de verificações de bombas

Este notebook oferece duas operações:

1. `tratar_csv_para_json(...)`: lê um CSV bruto, aplica o tratamento e grava um JSON tratado.
2. `combinar_jsons_tratados(...)`: acrescenta os registros de um JSON tratado ao JSON atual e salva uma base única.

A combinação mantém todos os registros. A coluna `REPETIDO` é recalculada sobre a base completa, como no fluxo original; nenhum registro é apagado automaticamente.

Os CSVs devem conter as colunas esperadas pelo tratamento original. Caminhos relativos são considerados a partir da pasta em que o notebook está sendo executado.


In [ ]:
import csv
import json
from pathlib import Path
import pandas as pd



import json
import re
import unicodedata
from collections import Counter
import pandas as pd


# ---------------------------------------------------------------------------
# Configurações
# ---------------------------------------------------------------------------
ARQ_JSON = "verificacoes_bombas_tratado.json"

COLUNAS_PRINCIPAIS = [
    "MUNICIPIO",
    "BAIRRO",
    "PROPRIETARIO",
    "DESCRICAO",
    "NUMERO_INMETRO",
    "NUMERO_SERIE",
    "DATA_VERIFICACAO",
    "RESULTADO",
]

COLUNAS_NECESSARIAS = COLUNAS_PRINCIPAIS + [
    "ENDERECO_COMPLETO", "RLD_ID", "NR_CGC_INDICE"
]

COLS_TEXTO = [
    "FISCAL", "MOTORISTA", "PROPRIETARIO", "DESCRICAO",
    "ENDERECO_COMPLETO", "MUNICIPIO", "BAIRRO", "RESULTADO"
]
COLS_ID_NUM = ["RLD_ID", "IDFISCAL", "IDMOTORISTA", "COD_PROPR", "CODITEM"]
COLS_REDUNDANTES = ["INDICEDOC", "ESPECIE", "REGIONAL"]
COLS_AUXILIARES = ["ENDERECO_COMPLETO", "NUMERO_SERIE_CHAVE"]
COLS_ABREVIACAO = ["PROPRIETARIO", "RUA_NUM", "BAIRRO", "MUNICIPIO"]
COLS_ACENTO = ["PROPRIETARIO", "RUA_NUM", "BAIRRO", "MUNICIPIO"]
EXCECOES_ACENTO = {"MARQUES", "SINHA"}


# ---------------------------------------------------------------------------
# Regras de padronização dos dados
# ---------------------------------------------------------------------------
MAPA_NOMES = {
    "POSTO SRRA DA BOCAINA": "POSTO SERRA DA BOCAINA",
    "POSTO CARREFFOUR": "POSTO CARREFOUR",
}

ABREVIACOES_RUA = {
    r"^DR\.?\s": "DOUTOR ",
    r"^DRA\.?\s": "DOUTORA ",
    r"^VER\.?\s": "VEREADOR ",
    r"^PRES\.?\s": "PRESIDENTE ",
    r"^PREF\.?\s": "PREFEITO ",
    r"^DEP\.?\s": "DEPUTADO ",
    r"^PROF\.?\s": "PROFESSOR ",
    r"^PROFA\.?\s": "PROFESSORA ",
    r"^ENG\.?\s": "ENGENHEIRO ",
    r"^CEL\.?\s": "CORONEL ",
    r"^CAP\.?\s": "CAPITÃO ",
    r"^GAL\.?\s": "GENERAL ",
    r"^MAL\.?\s": "MARECHAL ",
    r"^TEN\.?\s": "TENENTE ",
    r"^SEN\.?\s": "SENADOR ",
    r"^GOV\.?\s": "GOVERNADOR ",
    r"^AV\.?\s": "AVENIDA ",
    r"^AL\.?\s": "ALAMEDA ",
    r"^ROD\.?\s": "RODOVIA ",
}

ABREVIACOES_TEXTO = [
    (r"\bJD\b\.?\s*(?=\S)", "JARDIM "),
    (r"\bVL\b\.?\s*(?=\S)", "VILA "),
    (r"^V\.\s*(?=\S)", "VILA "),
    (r"\bPQ\b\.?\s*(?=\S)", "PARQUE "),
    (r"\bSTO\b\.?\s*(?=\S)", "SANTO "),
    (r"\bSTA\b\.?\s*(?=\S)", "SANTA "),
    (r"\bAV\b\.?\s*(?=\S)", "AVENIDA "),
    (r"\bCOM\.\s*(?=\S\S)", "COMÉRCIO "),
    (r"\bS\.?\s*J\.?\s*CAMPOS\b", "SÃO JOSÉ DOS CAMPOS"),
    (r"\b(\d+)\s+EST\b", r"\1 ESTRELAS"),
    (r"\bD\.?\s+(?=PEDRO\b)", "DOM "),
    (r"\bLTDA\s+EPP\b", "LTDA - EPP"),
]

CORRECOES_DIGITACAO = [
    (r"\bAUTU\b", "AUTO"),
    (r"\bANDROMERDA\b", "ANDROMEDA"),
    (r"\bVTRINE\b", "VITRINE"),
    (r"\bODOIL\b", "RODOIL"),
    (r"\bEUCALIPITOS\b", "EUCALIPTOS"),
    (r"\bENTREGAS RAPIDA\b", "ENTREGAS RAPIDAS"),
    (r"\bMARTINS DE SIQUEI\b", "MARTINS DE SIQUEIRA"),
]

MAPA_RESULTADO = {"Sem Verif.": "Sem Verificação"}

# Modelo: "RUA, NUM - BAIRRO - CIDADE - CEP: 00000000"
REGEX_ENDERECO = (
    r"^(?P<RUA_NUM>.*?(?:\d|S/N|SN))\s-\s(?P<BAIRRO>.*?)\s*-\s"
    r"(?P<CIDADE>(?:(?!\s-\s).)+?)\s-\sCEP:\s*(?P<CEP>\S*)$"
)
PALAVRA = re.compile(r"[^\W\d_]+")


# ---------------------------------------------------------------------------
# Funções auxiliares
# ---------------------------------------------------------------------------
def verificar_entrada(df):
    if not isinstance(df, pd.DataFrame):
        raise TypeError("A variável 'df' precisa ser um DataFrame do pandas.")

    if df.columns.duplicated().any():
        repetidas = df.columns[df.columns.duplicated()].tolist()
        raise ValueError(f"Há nomes de colunas duplicados: {repetidas}")

    faltando = [c for c in COLUNAS_NECESSARIAS if c not in df.columns]
    if faltando:
        raise ValueError(
            "O CSV não contém todas as colunas necessárias para o tratamento. "
            f"Colunas ausentes: {faltando}. Confira se o arquivo corresponde ao padrão de dados do IPEM."
        )


def vazios_para_nan(df):
    """Troca textos vazios/contendo só espaços por valores nulos."""
    return df.replace(r"^\s*$", pd.NA, regex=True)


def limpar_texto(s):
    """Remove espaços nas pontas e reduz espaços duplicados."""
    return s.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)


def chave(s):
    """Representação comparável sem acento/pontuação, em maiúsculas."""
    if pd.isna(s):
        return pd.NA
    s = unicodedata.normalize("NFKD", str(s))
    s = s.encode("ascii", "ignore").decode().upper()
    s = re.sub(r"['`´]", "", s)
    s = re.sub(r"[^A-Z0-9 ]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


def normalizar_serie(s):
    """Série em maiúsculas, sem espaço/pontuação."""
    return s.map(chave).astype("string").str.replace(" ", "", regex=False)


def completar_zeros(s, tamanho):
    return s.astype("string").str.zfill(tamanho)


def unificar_grafia(s):
    """Unifica grafias que diferem por acento/pontuação."""
    k = s.map(chave)
    cand = pd.DataFrame({"k": k, "v": s}).dropna().drop_duplicates()
    if cand.empty:
        return s
    freq = s.value_counts()
    cand["acentos"] = cand["v"].map(lambda v: sum(ord(ch) > 127 for ch in v))
    cand["freq"] = cand["v"].map(freq)
    melhor = (
        cand.sort_values(["acentos", "freq"], ascending=False)
        .drop_duplicates("k").set_index("k")["v"]
    )
    return k.map(melhor).where(s.notna())


def expandir_abreviacoes(s):
    if pd.isna(s):
        return s
    for padrao, texto in ABREVIACOES_TEXTO + CORRECOES_DIGITACAO:
        s = re.sub(padrao, texto, str(s))
    s = re.sub(r"\s+,", ",", s)
    return re.sub(r"\s+", " ", s).strip()


def unificar_palavras(df, colunas):
    """Unifica palavras que diferem por acento, mantendo exceções."""
    freq = Counter()
    for c in colunas:
        for texto, n in df[c].value_counts().items():
            for p in PALAVRA.findall(texto):
                freq[p] += n

    grupos = {}
    for p in freq:
        grupos.setdefault(chave(p), []).append(p)

    melhor = {
        k: max(v, key=lambda p: (sum(ord(ch) > 127 for ch in p), freq[p]))
        for k, v in grupos.items()
        if len(v) > 1 and k not in EXCECOES_ACENTO
    }
    trocar = lambda m: melhor.get(chave(m.group()), m.group())
    for c in colunas:
        df[c] = df[c].map(
            lambda t: t if pd.isna(t) else PALAVRA.sub(trocar, str(t))
        )
    return df


# ---------------------------------------------------------------------------
# Etapas do tratamento
# ---------------------------------------------------------------------------
def padronizar_datas(df):
    """Preserva DATA_VERIFICACAO; converte também DT_HORA_VERIF quando existir."""
    df["DATA_VERIFICACAO"] = pd.to_datetime(
        df["DATA_VERIFICACAO"], dayfirst=True, format="mixed", errors="coerce"
    )
    ano = df["DATA_VERIFICACAO"].dt.year

    if "DT_HORA_VERIF" in df.columns:
        df["DT_HORA_VERIF"] = pd.to_datetime(
            df["DT_HORA_VERIF"], dayfirst=True, format="mixed", errors="coerce"
        )
        ano = df["DT_HORA_VERIF"].dt.year.fillna(ano)

    df["ANO"] = ano.astype("Int64")
    return df


def separar_endereco(df):
    """Complementa endereço com RUA_NUM e CEP"""
    end = df["ENDERECO_COMPLETO"].astype("string").str.extract(REGEX_ENDERECO)
    df["CEP"] = completar_zeros(end["CEP"], 8)
    df["RUA_NUM"] = end["RUA_NUM"].str.upper().replace("", pd.NA)

    for padrao, texto in ABREVIACOES_RUA.items():
        df["RUA_NUM"] = df["RUA_NUM"].str.replace(padrao, texto, regex=True)
    regex_ok = end["CIDADE"].notna()
    df["BAIRRO"] = end["BAIRRO"].replace("", pd.NA).where(regex_ok, df["BAIRRO"])
    df["MUNICIPIO"] = end["CIDADE"].where(regex_ok, df["MUNICIPIO"])
    df["BAIRRO"] = df["BAIRRO"].str.upper()
    df["MUNICIPIO"] = df["MUNICIPIO"].str.upper()
    return df


def converter_ids(df):
    for c in COLS_ID_NUM:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce").astype("Int64")
    return df


def padronizar_textos(df):
    for c in COLS_ABREVIACAO:
        df[c] = df[c].map(expandir_abreviacoes)

    partes = df["RUA_NUM"].str.extract(r"^(?P<nome>.*?)(?P<num>,\s*[^,]*)?$")
    df["RUA_NUM"] = unificar_grafia(partes["nome"]) + partes["num"].fillna("")

    for c in ["BAIRRO", "MUNICIPIO"]:
        df[c] = unificar_grafia(df[c])

    df = unificar_palavras(df, COLS_ACENTO)
    df["PROPRIETARIO_CHAVE"] = df["PROPRIETARIO"].map(chave)
    return df


def tratar(df):
    """Aplica o tratamento completo ao DataFrame preparado a partir do CSV."""
    verificar_entrada(df)
    df = vazios_para_nan(df.copy())
    df = padronizar_datas(df)

    for c in COLS_TEXTO:
        if c in df.columns:
            df[c] = limpar_texto(df[c])

    for c in ["PROPRIETARIO", "BAIRRO", "MUNICIPIO"]:
        df[c] = df[c].str.upper()

    df["PROPRIETARIO"] = df["PROPRIETARIO"].replace(MAPA_NOMES)
    df["RESULTADO"] = df["RESULTADO"].replace(MAPA_RESULTADO)

    df["NUMERO_SERIE_CHAVE"] = normalizar_serie(df["NUMERO_SERIE"])
    df["NUMERO_SERIE"] = df["NUMERO_SERIE_CHAVE"]
    df["NR_CGC_INDICE"] = completar_zeros(df["NR_CGC_INDICE"], 8)

    df = separar_endereco(df)
    df = converter_ids(df)
    df = padronizar_textos(df)
    df = df.rename(columns={"SERVIÇO": "SERVICO"})

    df = df.drop(columns=COLS_REDUNDANTES + COLS_AUXILIARES, errors="ignore")

    # Mantém as colunas principais no início do arquivo.
    outras = [c for c in df.columns if c not in COLUNAS_PRINCIPAIS]
    return df[COLUNAS_PRINCIPAIS + outras]


def sinalizar_repetidos(df):
    """Marca RLD_ID + NUMERO_INMETRO repetidos, não exclui registros."""
    df = df.copy()
    chaves_validas = df["RLD_ID"].notna() & df["NUMERO_INMETRO"].notna()
    df["REPETIDO"] = chaves_validas & df.duplicated(
        ["RLD_ID", "NUMERO_INMETRO"], keep=False
    )
    return df

# Validação da base tratada
def validar(df):
    """Mostra indicadores de qualidade e distribuição por ano."""
    print("Formato:", df.shape)
    print("Datas inválidas:", df["DATA_VERIFICACAO"].isna().sum())
    print("Municípios nulos:", df["MUNICIPIO"].isna().sum())
    print("Bairros nulos:", df["BAIRRO"].isna().sum())
    print("Endereços sem CEP extraído:", df["CEP"].isna().sum())
    print("CEP com formato inválido:", df["CEP"].dropna().str.fullmatch(r"\d{8}").eq(False).sum())
    print("CNPJ (raiz) com formato inválido:",
          df["NR_CGC_INDICE"].dropna().str.fullmatch(r"\d{8}").eq(False).sum())
    print("Série com espaço/pontuação/minúscula:",
          df["NUMERO_SERIE"].str.contains(r"[^A-Z0-9]", na=False).sum())

    nome_rua = df["RUA_NUM"].str.replace(r",\s*[^,]*$", "", regex=True)
    for nome, col in [
        ("Nome da rua", nome_rua),
        ("BAIRRO", df["BAIRRO"]),
        ("MUNICIPIO", df["MUNICIPIO"]),
    ]:
        grafias = col.groupby(col.map(chave)).nunique()
        print(f"{nome} com mais de uma grafia:", (grafias > 1).sum())

    pendentes = 0
    for c in COLS_ABREVIACAO:
        s = df[c].fillna("")
        pendentes += (s.map(expandir_abreviacoes) != s).sum()
    print("Abreviações/correções pendentes:", pendentes)
    print("Registros marcados como REPETIDO:", df["REPETIDO"].sum())
    print("Anos:", df["ANO"].value_counts().sort_index().to_dict())


def exportar_json(df, caminho=ARQ_JSON):
    """Exporta JSON UTF-8 com datas ISO e nulos como null."""
    registros = json.loads(df.to_json(
        orient="records", date_format="iso", date_unit="s"
    ))
    with open(caminho, "w", encoding="utf-8") as f:
        json.dump(registros, f, ensure_ascii=False, indent=2)
    print("JSON salvo em:", caminho)

In [ ]:
# ---------------------------------------------------------------------------
# Leitura e preparação de CSV
# ---------------------------------------------------------------------------
def ler_csv_origem(caminho, separador=None, encoding="latin1"):
    """Lê CSV como texto e tenta detectar o separador quando não informado."""
    caminho = Path(caminho)
    if separador is None:
        with caminho.open("r", encoding=encoding, errors="replace", newline="") as f:
            amostra = f.read(8192)
        try:
            separador = csv.Sniffer().sniff(amostra, delimiters=";,\t").delimiter
        except csv.Error:
            separador = ";"
    dados = pd.read_csv(caminho, sep=separador, encoding=encoding, dtype=str)
    dados.columns = dados.columns.str.strip()

    renomear = {
        "NOMEFANTASIA": "PROPRIETARIO",
        "DESC_ITEM": "DESCRICAO",
        "NR_INMETRO": "NUMERO_INMETRO",
        "NR_SERIE": "NUMERO_SERIE",
        "DATAVERIFICACAO": "DATA_VERIFICACAO",
        "CIDADE": "MUNICIPIO",
    }
    dados = dados.rename(columns={k: v for k, v in renomear.items() if k in dados.columns})

    # No arquivo de origem, município e bairro podem estar embutidos no endereço.
    if "ENDERECO_COMPLETO" in dados.columns:
        if "MUNICIPIO" not in dados.columns:
            dados["MUNICIPIO"] = dados["ENDERECO_COMPLETO"].map(
                lambda x: str(x).strip().split(" - ")[-2].strip() if pd.notna(x) and len(str(x).split(" - ")) >= 2 else pd.NA
            )
        if "BAIRRO" not in dados.columns:
            dados["BAIRRO"] = dados["ENDERECO_COMPLETO"].map(
                lambda x: str(x).strip().split(" - ")[-3].strip() if pd.notna(x) and len(str(x).split(" - ")) >= 3 else pd.NA
            )
    return dados


def tratar_csv_para_json(caminho_csv, caminho_json="dados_tratados.json", separador=None, encoding="latin1"):
    """Lê um CSV de origem, aplica as regras do notebook e salva JSON tratado."""
    dados = ler_csv_origem(caminho_csv, separador=separador, encoding=encoding)
    tratado = sinalizar_repetidos(tratar(dados))
    exportar_json(tratado, caminho_json)
    return tratado


def selecionar_csv_e_tratar():
    """Abre seletor de arquivo no Colab ou Jupyter, trata o CSV e baixa/salva o JSON."""
    try:
        from google.colab import files
        enviado = files.upload()
        if not enviado:
            print("Nenhum arquivo selecionado.")
            return None
        nome_csv = next(iter(enviado))
        saida = "dados_tratados.json"
        resultado = tratar_csv_para_json(nome_csv, saida)
        files.download(saida)
        return resultado
    except ImportError:
        import tkinter as tk
        from tkinter import filedialog
        raiz = tk.Tk()
        raiz.withdraw()
        caminho = filedialog.askopenfilename(
            title="Selecione o arquivo CSV",
            filetypes=[("Arquivos CSV", "*.csv"), ("Todos os arquivos", "*.*")],
        )
        raiz.destroy()
        if not caminho:
            print("Nenhum arquivo selecionado.")
            return None
        pasta = str(Path(caminho).resolve().parent)
        saida = str(Path(pasta) / "dados_tratados.json")
        return tratar_csv_para_json(caminho, saida)


In [ ]:
# ---------------------------------------------------------------------------
# Combinação de dois JSONs já tratados
# ---------------------------------------------------------------------------
def _ler_json_tratado(caminho):
    with open(caminho, "r", encoding="utf-8") as arquivo:
        registros = json.load(arquivo)
    if not isinstance(registros, list) or any(not isinstance(item, dict) for item in registros):
        raise ValueError(f"O arquivo {caminho} precisa conter uma lista JSON de objetos/ registros.")
    return pd.DataFrame(registros)


def combinar_jsons_tratados(json_atual, json_novo, caminho_saida="verificacoes_bombas_tratado.json"):
    """Concatena dois JSONs tratados, recalcula REPETIDO e salva uma base única."""
    atual = _ler_json_tratado(json_atual)
    novo = _ler_json_tratado(json_novo)
    combinado = pd.concat([atual, novo], ignore_index=True, sort=False)

    # Recalcula os sinalizadores considerando registros dos dois arquivos.
    if {"RLD_ID", "NUMERO_INMETRO"}.issubset(combinado.columns):
        combinado = sinalizar_repetidos(combinado)
    else:
        raise ValueError("Os JSONs precisam conter as colunas RLD_ID e NUMERO_INMETRO para recalcular REPETIDO.")

    exportar_json(combinado, caminho_saida)
    print(f"Registros no arquivo atual: {len(atual)}")
    print(f"Registros acrescentados: {len(novo)}")
    print(f"Total na base atualizada: {len(combinado)}")
    return combinado


def selecionar_jsons_e_combinar():
    """Abre seletores para escolher a base atual e o JSON novo a acrescentar."""
    try:
        from google.colab import files
        print("Selecione primeiro o JSON atual do sistema.")
        atuais = files.upload()
        if not atuais:
            print("Nenhum arquivo selecionado.")
            return None
        arquivo_atual = next(iter(atuais))

        print("Agora selecione o JSON tratado que será acrescentado.")
        novos = files.upload()
        if not novos:
            print("Nenhum segundo arquivo selecionado.")
            return None
        arquivo_novo = next(iter(novos))
        saida = "verificacoes_bombas_tratado.json"
        resultado = combinar_jsons_tratados(arquivo_atual, arquivo_novo, saida)
        files.download(saida)
        return resultado
    except ImportError:
        import tkinter as tk
        from tkinter import filedialog
        raiz = tk.Tk()
        raiz.withdraw()
        arquivo_atual = filedialog.askopenfilename(
            title="Selecione o JSON atual do sistema",
            filetypes=[("Arquivos JSON", "*.json"), ("Todos os arquivos", "*.*")],
        )
        if not arquivo_atual:
            raiz.destroy()
            print("Nenhum arquivo selecionado.")
            return None
        arquivo_novo = filedialog.askopenfilename(
            title="Selecione o JSON tratado para acrescentar",
            filetypes=[("Arquivos JSON", "*.json"), ("Todos os arquivos", "*.*")],
        )
        raiz.destroy()
        if not arquivo_novo:
            print("Nenhum segundo arquivo selecionado.")
            return None
        saida = str(Path(arquivo_atual).resolve().parent / "verificacoes_bombas_tratado.json")
        return combinar_jsons_tratados(arquivo_atual, arquivo_novo, saida)


## Como usar

1. Execute as células de preparação acima, em ordem.
2. Para tratar um CSV, execute a célula **Selecionar CSV e tratar** no fim do notebook.
3. Para atualizar a base do sistema, execute **Selecionar dois JSONs e combinar**. Escolha primeiro o JSON atual e depois o JSON novo.
4. O arquivo resultante será `verificacoes_bombas_tratado.json`. No Colab, ele será baixado; no Jupyter, será salvo junto ao arquivo selecionado.

A junção acrescenta os registros e recalcula `REPETIDO`; não remove linhas duplicadas automaticamente.


## Seleção de arquivos

Execute a célula correspondente à operação desejada. Será aberta uma janela para selecionar os arquivos. No Google Colab, o seletor de envio será exibido na página.


In [ ]:
# Tratar um arquivo CSV e gerar o JSON tratado.
selecionar_csv_e_tratar()


In [ ]:
# Combinar a base JSON atual com outro JSON tratado.
selecionar_jsons_e_combinar()
